# NB03 – pandas: a planilha que não trava
**Métodos e Ferramentas de Data Science – FGV MBA · Aula 3**

## A pergunta de negócio
O diretor comercial de um e-commerce desconfia que **o atraso na entrega está derrubando a satisfação dos clientes**. Ele quer saber:

> **O atraso piora a avaliação? Em que estados o problema é maior?**

## Os dados
Base pública da **Olist**, e-commerce brasileiro, com cerca de 100 mil pedidos de 2016 a 2018 ([Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)). Os nomes das colunas estão em inglês, como no mundo real.

| Tabela | Uma linha é… | Colunas principais |
|---|---|---|
| `pedidos` | um pedido | `order_id`, `customer_id`, `order_status`, `order_purchase_timestamp` (compra), `order_delivered_customer_date` (entrega), `order_estimated_delivery_date` (prazo prometido) |
| `itens` | um item de um pedido | `order_id`, `product_id`, `price` (preço), `freight_value` (frete) |
| `clientes` | um cliente num pedido | `customer_id`, `customer_unique_id`, `customer_city`, `customer_state` (UF) |
| `avaliacoes` | uma avaliação | `order_id`, `review_score` (nota de 1 a 5) |
| `produtos` | um produto | `product_id`, `product_category_name` (categoria), peso e dimensões |

## O ecossistema de bibliotecas: quando você vai ouvir falar de cada uma

| Biblioteca | Para que serve | Nesta disciplina |
|---|---|---|
| **pandas** | Tabelas: carregar, filtrar, agrupar, juntar | Todas as aulas |
| **NumPy** | Cálculo numérico rápido; é a base do pandas | Por trás do pandas |
| **Matplotlib** e **seaborn** | Gráficos | Aulas 1, 4 e 5 |
| **scikit-learn** | Machine learning clássico: classificação, previsão, agrupamento | Demonstração da Aula 2 |
| **TensorFlow**, **Keras** e **PyTorch** | Deep learning (redes neurais): imagem, texto, áudio, modelos grandes | Não usamos aqui. Seu time vai citá-las em projetos com dados não estruturados |

## 1. Carregar as tabelas

In [ ]:
import pandas as pd
pd.set_option("display.float_format", "{:.2f}".format)

BASE = "https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/dados/"
pedidos = pd.read_csv(BASE + "olist_pedidos.csv.gz")
itens = pd.read_csv(BASE + "olist_itens.csv.gz")
clientes = pd.read_csv(BASE + "olist_clientes.csv.gz")
avaliacoes = pd.read_csv(BASE + "olist_avaliacoes.csv.gz")
produtos = pd.read_csv(BASE + "olist_produtos.csv.gz")
print("Tabelas carregadas!")

## 2. Inspecionar antes de analisar
Antes de qualquer conta, responda: quantas linhas? Que tipos? Onde faltam dados?

In [ ]:
print("pedidos:", pedidos.shape)
pedidos.head(3)

In [ ]:
pedidos.info()   # tipos de cada coluna e quantos valores não nulos

**Repare:** as datas vieram como texto (`object`). Para fazer contas com datas, precisamos convertê-las.

In [ ]:
# Quantos valores faltam em cada coluna?
pedidos.isna().sum()

## 3. Tipos e datas: transformar texto em data

In [ ]:
colunas_data = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in colunas_data:
    pedidos[c] = pd.to_datetime(pedidos[c])

# Novas colunas: dias até a entrega e se atrasou
pedidos["dias_entrega"] = (pedidos["order_delivered_customer_date"] - pedidos["order_purchase_timestamp"]).dt.days
pedidos["atrasou"] = pedidos["order_delivered_customer_date"] > pedidos["order_estimated_delivery_date"]
pedidos[["order_purchase_timestamp", "order_delivered_customer_date", "dias_entrega", "atrasou"]].head()

## 4. Faltantes: por que falta a data de entrega?
Faltante não é só um buraco: muitas vezes é **informação**. Vamos ver o status dos pedidos sem data de entrega.

In [ ]:
sem_entrega = pedidos[pedidos["order_delivered_customer_date"].isna()]
sem_entrega["order_status"].value_counts()

A maioria são pedidos **não entregues** (enviados, cancelados, indisponíveis). Para medir prazo e atraso, faz sentido usar só os **entregues**.
Mas atenção: se a pergunta fosse "quantos clientes ficaram sem produto?", esses faltantes seriam o centro da análise.

## 5. Filtrar e ordenar

In [ ]:
entregues = pedidos[(pedidos["order_status"] == "delivered") & pedidos["order_delivered_customer_date"].notna()].copy()
print("Pedidos entregues:", len(entregues))

# As 5 entregas mais demoradas
entregues.sort_values("dias_entrega", ascending=False)[["order_id", "dias_entrega", "atrasou"]].head()

## 6. Resumir: estatística descritiva

In [ ]:
entregues["dias_entrega"].describe()

In [ ]:
print(f"Prazo médio: {entregues['dias_entrega'].mean():.1f} dias | mediana: {entregues['dias_entrega'].median():.0f} dias")
print(f"Pedidos entregues com atraso: {entregues['atrasou'].mean():.1%}")

**Pare e pense:** por que a média é maior que a mediana? (Guarde a resposta: é o tema da Aula 4.)

## 7. Juntar tabelas (`merge`) e agrupar (`groupby`)
O estado do cliente está na tabela `clientes`. Juntamos pela chave `customer_id`, como um **PROCV** do Excel.

In [ ]:
entregues = entregues.merge(clientes[["customer_id", "customer_state"]], on="customer_id", how="left")

por_uf = (entregues.groupby("customer_state")
          .agg(pedidos=("order_id", "count"), prazo_medio=("dias_entrega", "mean"), pct_atraso=("atrasou", "mean"))
          .query("pedidos >= 300")                        # só estados com volume suficiente
          .sort_values("pct_atraso", ascending=False))
por_uf.head(10)

### Cuidado: juntar pode duplicar linhas
Um pedido pode ter vários itens. Veja o que acontece com o número de linhas ao juntar `pedidos` com `itens`:

In [ ]:
print("pedidos:", len(pedidos), "→ pedidos + itens:", len(pedidos.merge(itens, on="order_id")))

A tabela cresceu porque **cada item vira uma linha**. Se você somar algo do pedido depois disso, conta em dobro.
**Regra prática:** antes de juntar, **agregue** a tabela do lado "muitos" para o nível que você quer analisar.
As avaliações têm o mesmo risco (alguns pedidos têm mais de uma). Então primeiro calculamos a nota média **por pedido**:

In [ ]:
nota_por_pedido = avaliacoes.groupby("order_id")["review_score"].mean().rename("nota")
entregues = entregues.merge(nota_por_pedido, on="order_id", how="left")

# A pergunta do diretor: o atraso piora a nota?
entregues.groupby("atrasou")["nota"].mean()

In [ ]:
# Distribuição das notas: atrasados × no prazo (em % de cada grupo)
pd.crosstab(entregues["atrasou"], entregues["nota"].round(), normalize="index").round(2)

---
## 8. Sua vez: exercícios individuais
Responda no seu Colab. O gabarito sai depois da aula.

**Exercício 1.** Que porcentagem de **todos** os pedidos tem status `delivered`?
*Dica:* `value_counts(normalize=True)` ou a média de uma comparação.

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 2.** Qual a taxa de atraso dos pedidos entregues a clientes do **RJ**?

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 3.** Qual estado tem o **maior prazo médio** de entrega, entre os que têm pelo menos 300 pedidos entregues?
*Dica:* use a tabela `por_uf` e ordene por outra coluna.

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 4.** Quais as **3 categorias de produto que mais faturaram** (soma de `price`)?
*Dica:* junte `itens` com `produtos` pela chave `product_id`, agrupe por `product_category_name`, some `price` e ordene.

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 5.** Quantos **clientes únicos** existem? Compare `customer_id` com `customer_unique_id`. Por que os números são diferentes?

In [ ]:
# SEU CÓDIGO AQUI


---
## 9. Resposta ao diretor (e o que ela não diz)
**Achado:** pedidos atrasados têm nota média bem menor do que os entregues no prazo (cerca de 2,6 contra 4,3). As maiores taxas de atraso estão no Nordeste (AL, MA, PI, CE, SE, BA). Em volume, o RJ pesa mais: 13% de atraso sobre mais de 12 mil pedidos.

**Limitações:**
1. **Correlação não é causa:** os estados com mais atraso também são mais distantes e podem ter outros problemas (frete, produto).
2. **Quem avalia:** nem todo cliente responde; se os insatisfeitos respondem mais (ou menos), a média muda.
3. **Período:** dados de 2016 a 2018; a operação de hoje pode ser outra.
4. **Faltantes:** excluímos pedidos não entregues, que provavelmente geram as piores experiências.

---
## Erros comuns e como resolver

| Mensagem | O que aconteceu | O que fazer |
|---|---|---|
| `KeyError: 'customer_state'` | A coluna não existe nessa tabela (ainda não houve o `merge`) | Execute a célula do `merge` antes |
| `TypeError: unsupported operand type(s) for -: 'str' and 'str'` | Datas ainda como texto | Execute a seção 3 (`pd.to_datetime`) |
| Valores dobrados depois de um `merge` | Junção de tabela "muitos" sem agregar | Agregue antes de juntar (seção 7) |
| `MergeError` ou colunas `_x` e `_y` | Mesma coluna nas duas tabelas | Selecione só as colunas necessárias antes do `merge` |
| A célula demora | Base com 100 mil linhas | Normal: espere alguns segundos |

## Plano B
Se o download falhar, baixe os 5 arquivos `olist_*.csv.gz` da pasta `dados` do repositório (link no ECLASS), arraste para o painel de arquivos do Colab e troque a primeira linha da seção 1 por `BASE = ""`.